# 4. Overfitting

Phần này khuyến khích dùng GPU trên colab hoặc kaggle do thời gian train tương đối dài (10000 epoch)

# 4.1. What is it?

Nếu ta nhớ lại thì bài toán Machine Learning nói chung là một bài toán tối ưu, trong đó ta tối ưu tham số của mô hình sao cho hàm mất mát trên tập dữ liệu huấn luyện là nhỏ nhất. Điều này dẫn đến một vấn đề, đó là hoạt động của mô hình chịu ảnh hưởng bởi quan hệ giữa số mẫu của tập dữ liệu và số tham số có thể tối ưu.

Xét bài toán phân lớp nhị phân đơn giản với đầu vào gồm 2 đặc trưng $x_0$ và $x_1$. Đối với cả 2 lớp phân phối dữ liệu thực tế là phân phối Gauss độc lập 2 chiều, trong đó lớp '0' có kỳ vọng bằng [0,0] và phương sai bằng [1,1], còn lớp '1' có kỳ vọng bằng [1,1] và phương sai bằng [1,1].

Dữ liệu huấn luyện sẽ chỉ gồm 30 mẫu còn dữ liệu huấn luyện có 200 mẫu. Mô hình được sử dụng sẽ là mô hình LinearNN giống ở phần bài tập trước với L=4, layers_dims=[2,50,50,50,1]. 

Bài 4.1: Chạy mô hình, xem và nhận xét kết quả accuracy và loss của tập train và tập test khi epoch tăng dần.

In [ ]:
from utils import *

torch.manual_seed(1)
np.random.seed(1)

class LinearNN_4(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(in_features=2, out_features=50),
            nn.ReLU(inplace=True),
            nn.Linear(in_features=50, out_features=50),
            nn.ReLU(inplace=True),
            nn.Linear(in_features=50, out_features=50),
            nn.ReLU(inplace=True),
            nn.Linear(in_features=50, out_features=1),
            nn.Sigmoid()
        )

    def forward(self,x):
        return self.model(x)

In [ ]:
param = []
param.append(
    {
        'mean': [0, 0],
        'cov': [[1,0],[0,1]],
        'label': 0
    }
)

param.append(
    {
        'mean': [1, 1],
        'cov': [[1,0],[0,1]],
        'label': 1
    }
)

train_data = get_syn_dataset(30, param, func_type='gauss')
test_data = get_syn_dataset(200, param, func_type='gauss')
num_epochs = 10000
batch_size = 10
lr = 1e-3

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=SynCollate())
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=True, collate_fn=SynCollate())

model = LinearNN_4()

BCELoss = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr = lr)
trained_model, acc_list = trainer(model, train_loader, test_loader, num_epochs, batch_size, optimizer, BCELoss, n_epoch_to_log=1000, bin=True, cal_test_loss=True)
print(f'NN without anything: Train_acc: {acc_list[-1][0]:.2f}%. Test_acc: {acc_list[-1][1]:.2f}%')
plot_curve(acc_list,'both')
plot_decision_boundary(train_data.x.numpy(), train_data.y.numpy(), model)
plot_decision_boundary(test_data.x.numpy(), test_data.y.numpy(), model)

Hiện tượng trên được gọi là overfitting, đó là khi một mô hình có thể hoạt động rất tốt khi đánh giá trên tập train nhưng có kết quả kém khi thực hiện trên tập test. 

Giải thích một cách trực giác thì khi mô hình đủ phức tạp (số lớp đủ sâu, số neural lớn) thì mô hình khả năng học thuộc toàn bộ dữ liệu huấn luyện thay vì trích xuất các đặc trưng giúp giải quyết bài toán phân lớp.

Giải thích một cách toán học thì bài toán học máy thực chất là một bài toán xấp xỉ hàm (tuyến tính hoặc phi tuyến) với M tham số tự do, các tham số này sẽ bị ràng buộc bởi N mẫu trong tập train. Khi mà mô hình phức tạp và số lượng mẫu ít thì M >> N, lúc này mô hình có thể thay đổi vô cùng linh hoạt để hàm xấp xỉ đi qua mọi điểm mẫu (VD dùng hàm bậc 2 với 3 tham số để ước lượng hàm đi qua 2 điểm mẫu thì sẽ có vô số lời giải), điều này dẫn đến mô hình có thể trượt xa khỏi hàm thực tế.

Để giải quyết vấn đề trên thì ta có thể sử dụng nhiều phương pháp. Ở đây giới thiệu Dropout và Regularization.

# 4.2. Dropout và Regularization

Dropout là phương pháp loại bỏ kết quả của một số neural ngẫu nhiên tại các lớp, tức là các neural này sẽ không được xét đến khi tính đầu ra. Tại sao lại như vậy? Bằng việc loại bỏ ngẫu nhiên các neural, ta đang tạo thêm nhiễu cho mô hình. Lúc này mô hình sẽ không thể học thuộc các mẫu nữa do cùng mẫu thì với các epoch khác nhau sẽ có đặc trưng khác nhau.

Việc triển khai Dropout sẽ được thực hiện thông qua nn.Dropout(p) trong đó p là xác suất bị bỏ qua của mỗi neural trong lớp đó (có thể hiểu là giữ lại khoảng (1-p)*100% neural, p=0 tương đương với không dùng Dropout).

Bài tập 4.2: Thay đổi p ở các lớp, train mô hình và đưa ra nhận xét về kết quả.

In [ ]:
class LinearNN_Dropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(in_features=2, out_features=50),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(in_features=50, out_features=50),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(in_features=50, out_features=50),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(in_features=50, out_features=1),
            nn.Sigmoid()
        )

    def forward(self,x):
        return self.model(x)

In [ ]:
model_dropout = LinearNN_Dropout()

BCELoss = nn.BCELoss() 
optimizer_dropout = optim.SGD(model_dropout.parameters(), lr = lr)
trained_model, acc_list = trainer(model_dropout, train_loader, test_loader, num_epochs, batch_size, optimizer_dropout, BCELoss, n_epoch_to_log=1000, bin=True, cal_test_loss=True)
print(f'NN with dropout: Train_acc: {acc_list[-1][0]:.2f}%. Test_acc: {acc_list[-1][1]:.2f}%')
plot_curve(acc_list,'both')
plot_decision_boundary(train_data.x.numpy(), train_data.y.numpy(), model_dropout)
plot_decision_boundary(test_data.x.numpy(), test_data.y.numpy(), model_dropout)

Regularization sử dụng thêm một hàm loss để tạo ràng buộc về độ lớn của các tham số. Gọi $y$ là phân lớp thực tế, $\hat{y}$ là phân lớp mà mô hình dự đoán, w là vector chứa các tham số của mô hình, hàm loss khi sử dụng Regularization sẽ trông như sau:

$$ L_{reg}(y,\hat{y},w) = L(y,\hat{y}) + \lambda ||w||_p $$

$||\cdot||_p > 0$ là norm p của vector, $\lambda$ là hyperparamter dùng để xác định mức ảnh hưởng của độ lớn tham số khi tính loss. Việc cộng thêm norm p làm cho tham số của mô hình không được quá lớn, như vậy thì các tham số đều sẽ nhỏ đi, trong đó các tham số ít quan trọng trong việc dự đoán mẫu sẽ bị giảm nhiều hơn (tiến về 0). Điều này giống như việc ta đang giảm bậc của mô hình một cách liên tục, VD thay vì giảm trực tiếp từ hàm bậc 2 $ax^2 + bx + c$ xuống còn $bx + c$ thì ta sử dụng hàm 

$$ \frac{a}{100} x^2 + \frac{b}{2} x + c $$
 
Hai loại norm thường gặp trong Regularization là norm 1 và norm 2, hoặc còn có thể được biết đến là LASSO và Ridge. Do norm 1 không tồn tại đạo hàm tại 0 nên trong các hệ thống norm 2 được sử dụng phổ biến nhất. Trong mạng neural thì Regularization được biết đến với tên weight decay (suy giảm trọng số).

Việc triển khai Regularization sẽ được thực hiện thông qua optimizer (pytorch sử dụng norm 2 để tính)

Bài tập 4.3: Thay đổi weight decay, train mô hình và đưa ra nhận xét về kết quả.

In [ ]:
model_reg = LinearNN_4()

BCELoss = nn.BCELoss() 
optimizer_reg = optim.SGD(model_reg.parameters(), lr = lr, weight_decay=0.05)

trained_model, acc_list = trainer(model_reg, train_loader, test_loader, num_epochs, batch_size, optimizer_reg, BCELoss, n_epoch_to_log=1000, bin=True, cal_test_loss=True)
print(f'NN with regularization: Train_acc: {acc_list[-1][0]:.2f}%. Test_acc: {acc_list[-1][1]:.2f}%')
plot_curve(acc_list,'both')
plot_decision_boundary(train_data.x.numpy(), train_data.y.numpy(), model_reg)
plot_decision_boundary(test_data.x.numpy(), test_data.y.numpy(), model_reg)